# VarChat — 14B hücreleri: model dosyaları + 56 cevap

Bu defter dört iş yapar:
1. **`qwen14-taban`** (eğitimsiz 14B): Qwen2.5-14B-Instruct, birleştirme OLMADAN, GGUF + Q4_K_M. 7B'deki `qwen-taban`ın 14B karşılığı.
2. **`varchat-14b`** (eğitilmiş 14B): aynı taban + `varchat-lora-14b` adaptörü, kalıcı birleştirme, GGUF + Q4_K_M.
3. İkisi Ollama'ya, hazır `qwen2.5:14b` modelinin sohbet şablonuyla yüklenir (yalnız `FROM` satırı değişir).
4. İkisi de aynı 56 girdiye, aynı donmuş kaynaklarla (`arama_2026-10-05.jsonl`) cevap yazar (`e04`).
   Yargıç bu defterde YOK; puanlama yerelde `evaluation/e13_judge_14b.py` ile yapılır.

**Önce:** `fine_tune/f03_colab_train_14b.ipynb` bitmiş ve adaptörü Drive'a kopyalamış olmalı
(Drive'da `varchat-14b/varchat-lora-14b/drive_kopyasi.json` dosyası var).

**Nasıl açılır ve çalıştırılır (Colab'ın sitesi):**
1. colab.research.google.com → **Dosya → Not defteri yükle** (File → Upload notebook) → bilgisayarındaki `evaluation/e12_colab_14b.ipynb`.
2. **Çalışma zamanı → Çalışma zamanı türünü değiştir** → **L4 GPU**; "Yüksek RAM" (High-RAM) görünürse aç → Kaydet.
   L4 yoksa A100 de olur (saatte daha çok birim yakar). **T4 olmaz:** GPU belleği ve RAM yetmez, defter 0. bölümde durur.
3. **Çalışma zamanı → Tümünü çalıştır.** 1. bölümde Google Drive izni sorulur: kendi hesabınla izin ver.

**Süre:** L4'te ~2,5-3 saat: taban model indirme ~10 dk, iki dönüştürme ~20-25'er dk, birleştirme ~20 dk, iki model × 56 cevap ~20-30'ar dk.
Tarayıcı sekmesi açık kalsın, bilgisayar uykuya geçmesin. Bağlantı koparsa yeniden **Tümünü çalıştır**: yazılmış cevaplar Drive'da durduğu için
tekrar üretilmez (`e04` kaldığı yerden devam eder); model dosyaları ise sunucuyla birlikte silindiği için yeniden üretilir (~1 saat).
**Bitince:** Drive'daki **`varchat-14b/olcum_14b_sonuclari.zip`** dosyasını indir, projede `evaluation/results/` klasörüne aç.
Sonra **Çalışma zamanı → Bağlantıyı kes ve çalışma zamanını sil** (birim harcamasın).

**7B ile aynı tutulanlar** (karşılaştırılabilirlik için):
- llama.cpp **b11430** (7B model dosyalarını üreten sürüm) ve Ollama **0.35.1** (7B cevaplarını üreten sürüm) sabitlenir.
- Birleştirme işlemcide bf16 ile; dönüştürme f16 → Q4_K_M; transformers/peft eğitimdeki sürümler.
- Cevaplar `e04` ile: m01'in sistem metni ve ayarları (sıcaklık 0, tohum 42, num_ctx 8192, num_predict 1024). 1. bölüm, bugünkü kodun
  hazır 14B'nin Ekim'de aldığı istemleri 56/56 aynı ürettiğini kontrol eder; tutmazsa defter durur.

**Kart:** cevaplar L4'te üretilir; 7B'nin cevapları da L4'te, hazır 14B'ninkiler A100'de üretilmişti.
İki 14B hücresi aynı oturumda, aynı kartta üretilir; kartın adı kaydedilir.

**Sonuçlar:** Drive'da **`varchat-14b/results/`** klasörüne yazılır (7B'deki e07 gibi: repodaki sonuç dosyalarının kopyaları + YENİ dosyalar).
7B'nin `varchat-olcum` ve `varchat-gguf` klasörlerine dokunulmaz. 14. bölüm repodaki dosyalardan hiçbiri değişmemiş mi diye git'in kayıtlarıyla
karşılaştırır ve yalnız yeni dosyaları tek zip'e koyar.

## Ayar

In [ ]:
VARCHAT = "varchat-14b"                                    # egitilmis 14B (Ollama adi ve sonuc dosyalarindaki ad)
TABAN = "qwen14-taban"                                     # egitimsiz 14B, ayni hattan (birlestirme yok)
HAZIR = "qwen2.5:14b"                                      # Ollama'nin hazir 14B'si (Ekim'de olculen); sohbet sablonu buradan alinir
TABAN_HF_ID = "Qwen/Qwen2.5-14B-Instruct"
DRIVE_KOK = '/content/drive/MyDrive/varchat-14b'           # f03 ile ayni Drive klasoru (7B'nin klasorlerinden ayri)
LORA_DIR = f'{DRIVE_KOK}/varchat-lora-14b'                 # f03'un Drive'a kopyaladigi adaptor
SONUC_DRIVE = f'{DRIVE_KOK}/results'                       # cevaplar buraya yazilir (kopmadan sonra e04 kaldigi yerden devam eder)
ZIP = f'{DRIVE_KOK}/olcum_14b_sonuclari.zip'               # yalniz yeni dosyalar: bilgisayara indirilecek tek dosya

LLAMA_CPP_ETIKET, LLAMA_CPP_COMMIT = "b11430", "8345f333"   # 7B model dosyalarini ureten surum (gguf_ortam_2026-10-05.json)
OLLAMA_SURUM = "0.35.1"                                     # 7B cevaplarini ureten surum (ortam_2026-10-05.json)
SNAPSHOT = "arama_2026-10-05.jsonl"                         # donmus kaynaklar (uc modelin de kullandigi)
SNAPSHOT_SHA256 = "4acd9305274648670ecdbb43e8b10d65f7e2dfbeaab9d170b804da21e0c60f14"   # GitHub'daki (LF) surumun izi
KARSI = "uretim_qwen2.5-14b_2026-10-06.jsonl"               # hazir 14B'nin Ekim cevaplari: istemler bununla birebir ayni olmali
AYARLAR_BEKLENEN = {"num_ctx": 8192, "num_predict": 1024, "temperature": 0, "seed": 42, "repeat_penalty": 1.05}   # m01.AYARLAR
SABLON_7B_SHA256 = "ad191f154b3a23521d7b5b18f2d554cd7dba7bddbdd62782d7b712fceec8366f"  # qwen2.5:7b sablonu (FROM ile LICENSE arasi)

assert VARCHAT not in ("varchat", "varchat-v2") and TABAN != "qwen-taban", "7B modellerinin adlari kullanilmaz"
assert DRIVE_KOK.rstrip("/").split("/")[-1] not in ("varchat-lora", "varchat-lora-v2", "varchat-olcum", "varchat-gguf"), \
    "7B'nin Drive klasorlerine yazilmaz; DRIVE_KOK'a yeni bir ad ver"

## 0) GPU, RAM ve disk

Ollama'nın 14B modeli GPU'da ~12 GB ister; GPU'da 16 GB'tan az boş yer varsa (T4 gibi) defter burada durur.
RAM (birleştirme için) ve disk 3. bölümde, iş başlamadan önce kontrol edilir.

In [ ]:
import os, shutil, subprocess
!nvidia-smi --query-gpu=name,memory.total,memory.used --format=csv
KART = subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"], capture_output=True, text=True).stdout.strip()
assert KART, "GPU yok: Çalışma zamanı → Çalışma zamanı türünü değiştir → L4 GPU, sonra Tümünü çalıştır."
BOS_GPU_GB = int(subprocess.run(["nvidia-smi", "--query-gpu=memory.free", "--format=csv,noheader,nounits"],
                                capture_output=True, text=True).stdout.split()[0]) / 1024
RAM_GB = os.sysconf('SC_PAGE_SIZE') * os.sysconf('SC_PHYS_PAGES') / 1e9
print(f"kart: {KART} | GPU'da boş: {BOS_GPU_GB:.0f} GB | RAM: {RAM_GB:.0f} GB | boş disk: {shutil.disk_usage('/content').free / 1e9:.0f} GB")
assert BOS_GPU_GB >= 16, (f"GPU'da yalnız {BOS_GPU_GB:.0f} GB boş: Çalışma zamanı türünü L4 (ya da A100) yap, "
                          "sonra Tümünü çalıştır.")

## 1) Projeyi getir, Drive'a bağlan, paketler, istem kontrolü

Sonuç klasörü (`evaluation/results`) 7B'deki e07 gibi Drive'daki `varchat-14b/results` klasörüne bağlanır: repodaki sonuç dosyaları oraya
kopyalanır (Drive'da aynı adla bir dosya varsa üzerine yazılmaz), cevaplar oraya yazılır.
Kod sürümü **içerikten** doğrulanır: bugünkü `e04` + `m01`, hazır 14B'nin Ekim'de aldığı istemleri (sistem metni + soru) 56 girdinin
56'sında harfi harfine aynı üretmeli; üretim ayarları da beklenen değerler olmalı. Biri tutmazsa defter durur.

In [ ]:
import os
if not os.path.exists('/content/varchat'):
    !git clone -q https://github.com/atu-ce/varchat.git /content/varchat
else:
    # 'results' Drive'a baglandigi (sembolik bag) icin 'git pull' takilabilir: kodu guncelle, results'a dokunma (e07 ile ayni)
    !cd /content/varchat && git fetch -q origin && git checkout -q origin/main -- . ':(exclude)evaluation/results'
%cd /content/varchat
KOD = !git -C /content/varchat log --oneline -1 --no-color origin/main
print("kod:", KOD[0])
# Yereldeki surumlerle ayni istemci; requests Colab'in kendi surumu kalir (Colab paketleri ona bagli)
!pip install -q "ollama==0.6.2" "rapidfuzz==3.14.5"

In [ ]:
import shutil
from google.colab import drive
drive.mount('/content/drive')
os.makedirs(SONUC_DRIVE, exist_ok=True)

yerel = '/content/varchat/evaluation/results'
if not os.path.islink(yerel):
    for ad in os.listdir(yerel):
        kaynak = os.path.join(yerel, ad)
        if os.path.isfile(kaynak) and not os.path.exists(os.path.join(SONUC_DRIVE, ad)):
            shutil.copy2(kaynak, SONUC_DRIVE)          # Drive'da olanin ustune yazilmaz (onceki oturumun cevaplari korunur)
    shutil.rmtree(yerel)
    os.symlink(SONUC_DRIVE, yerel)
print("sonuçlar ->", SONUC_DRIVE, "|", len(os.listdir(yerel)), "dosya | Drive'da boş yer ~",
      round(shutil.disk_usage('/content/drive/MyDrive').free / 1e9, 1), "GB")
onceki = sorted(a for a in os.listdir(yerel) if a.startswith((f"uretim_{TABAN}_", f"uretim_{VARCHAT}_")))
if onceki:
    print("önceki oturumdan kalan cevap dosyaları (e04 bunlara kaldığı yerden devam eder):", onceki)

In [ ]:
import os, sys, json, hashlib
sys.path[:0] = ['/content/varchat', '/content/varchat/evaluation']
import e04_generate as e04

SONUC = '/content/varchat/evaluation/results'     # yeni cevap dosyalari repodaki dosyalarin yanina YENI adlarla yazilir
yol = os.path.join(SONUC, SNAPSHOT)
# Satir sonlari LF'ye cevrilip izlenir (Windows kopyasi CRLF, Colab kopyasi LF; icerik ayni olmali)
assert hashlib.sha256(open(yol, "rb").read().replace(b"\r\n", b"\n")).hexdigest() == SNAPSHOT_SHA256, f"{SNAPSHOT} Ekim'deki dosyayla aynı değil"
satirlar = [json.loads(l) for l in open(yol, encoding="utf-8")]
karsi = {json.loads(l)["girdi"]: json.loads(l) for l in open(os.path.join(SONUC, KARSI), encoding="utf-8")}
ayni = sum((karsi[r["girdi"]]["sistem"], karsi[r["girdi"]]["soru"]) == e04.istem(r) for r in satirlar)
print(f"istem kontrolü: {ayni}/{len(satirlar)} girdi, hazır 14B'nin Ekim istemiyle aynı | m01.AYARLAR = {e04.app.AYARLAR}")
assert ayni == len(satirlar) == 56, "Kod Ekim'deki istemleri üretmiyor: devam etme, çıktıyı Claude'a gönder"
assert e04.app.AYARLAR == AYARLAR_BEKLENEN, "m01 üretim ayarları beklenenden farklı: devam etme, çıktıyı Claude'a gönder"
# Denetim testleri (model çağırmaz, birkaç saniye)
!python evaluation/e10_check_tests.py | tail -1

## 2) Adaptör ve donmuş kaynak

Adaptör Drive'daki f03 kopyasından okunur. Önce `drive_kopyasi.json` aranır (kopyanın bittiğini gösterir) ve adaptörün parmak izi
o listedekiyle karşılaştırılır. `e04` sonuç klasöründeki en yeni arama dosyasını seçtiği için klasörde yalnız `arama_2026-10-05.jsonl` olmalı.

In [ ]:
import glob

def sha256(yol):
    h = hashlib.sha256()
    with open(yol, "rb") as f:
        for parca in iter(lambda: f.read(1 << 24), b""):
            h.update(parca)
    return h.hexdigest()

aramalar = sorted(os.path.basename(x) for x in glob.glob(f"{SONUC}/arama_*.jsonl"))
assert aramalar == [SNAPSHOT], f"e04 en yeni arama dosyasını seçer; klasörde yalnız {SNAPSHOT} olmalı: {aramalar}"

for ad in ("drive_kopyasi.json", "adapter_model.safetensors", "adapter_config.json"):
    assert os.path.exists(f"{LORA_DIR}/{ad}"), (f"Drive'da {LORA_DIR}/{ad} yok: f03 bitmemiş ya da Drive'a kopyası tamamlanmamış "
                                                "(f03'ün 6. bölümü). Durumu Claude'a yaz.")
KOPYA = json.load(open(f"{LORA_DIR}/drive_kopyasi.json"))
ADAPTOR_SHA = sha256(f"{LORA_DIR}/adapter_model.safetensors")
assert ADAPTOR_SHA == KOPYA["dosyalar"]["adapter_model.safetensors"], \
    "Drive'daki adaptör f03'ün kopyaladığı dosyayla aynı değil (parmak izi farklı): durumu Claude'a yaz."
print("adaptör: Drive kopyası tam (", KOPYA["tarih"], "| kod", KOPYA.get("kod"), ")")
LORA_AYAR = json.load(open(f"{LORA_DIR}/adapter_config.json"))
print("adaptör:", {k: LORA_AYAR.get(k) for k in ("base_model_name_or_path", "r", "lora_alpha", "lora_dropout")})
assert LORA_AYAR["base_model_name_or_path"] == TABAN_HF_ID, f"Bu adaptör {TABAN_HF_ID} için eğitilmemiş"

## 3) Hangi model dosyaları üretilecek?

Bu oturumda daha önce üretilmiş bir dosya, parmak izi (sha256) üretildiği an yazılan kayıtla aynıysa yeniden üretilmez.
Yeni oturumda model dosyaları yoktur (sunucuyla silinir), ikisi de yeniden üretilir.

In [ ]:
from huggingface_hub.constants import HF_HUB_CACHE

os.makedirs('/content/gguf', exist_ok=True)
YEREL = {ad: f"/content/gguf/{ad}-Q4_K_M.gguf" for ad in (TABAN, VARCHAT)}
BILGI = {ad: f"{SONUC}/gguf_ortam_14b_{ad}.json" for ad in YEREL}
URET = []
for ad in YEREL:
    hazir = (os.path.exists(YEREL[ad]) and os.path.exists(BILGI[ad])
             and json.load(open(BILGI[ad]))["sha256"] == sha256(YEREL[ad]))
    print(f"{ad}: {'hazır (parmak izi kayıtla aynı)' if hazir else 'üretilecek'}")
    if not hazir:
        URET.append(ad)
TABAN_INDIRILMIS = os.path.isdir(os.path.join(HF_HUB_CACHE, "models--" + TABAN_HF_ID.replace("/", "--")))
if URET:
    if VARCHAT in URET and not os.path.exists('/content/varchat-hf/config.json'):
        import psutil
        bos_ram = psutil.virtual_memory().available / 1e9
        assert bos_ram >= 38, (f"Kullanılabilir RAM {bos_ram:.0f} GB: 14B birleştirme ~35 GB ister. Çalışma zamanı türünde 'Yüksek RAM'i aç "
                               "(ya da A100 seç), sonra Tümünü çalıştır.")
    gereken = 60 if TABAN_INDIRILMIS else 90
    assert shutil.disk_usage('/content').free / 1e9 >= gereken, \
        f"En az ~{gereken} GB boş disk gerekir (ara dosya 29,5 GB + iki model dosyası ~9'ar GB" + ("" if TABAN_INDIRILMIS else " + taban model 29,5 GB") + ")."
    print("taban model sunucuda:", "var (bu oturumda indirilmiş)" if TABAN_INDIRILMIS else "yok, indirilecek")

## 4) Araçlar: transformers/peft (eğitimdeki sürümler) + llama.cpp b11430

In [ ]:
if URET:
    # Egitimdekiyle ayni surumler (adaptor bu surumlerle kaydedildi)
    !pip install -q "transformers==5.17.0" "peft==0.21.0" "accelerate==1.15.0" sentencepiece "protobuf>=4.21,<5"
    # Colab'in hazir kurulu eski torchao'su peft 0.21'in birlestirme yolunu bozuyor (5 Ekim Colab hatasi); bu is icin gerekmez
    !pip uninstall -y -q torchao
    !apt-get -qq install -y cmake libcurl4-openssl-dev > /dev/null
    if not os.path.exists('/content/llama.cpp'):
        !git clone -q --depth 1 --branch {LLAMA_CPP_ETIKET} https://github.com/ggml-org/llama.cpp /content/llama.cpp
    LLAMA_COMMIT = !git -C /content/llama.cpp log --oneline -1 --no-color
    # Tam kimlikle karsilastir: --depth 1 klonda 'log --oneline' kimligi 7 harfe kisaltir (8345f33), 7B'nin tam klonunda 9 harfti
    LLAMA_TAM = !git -C /content/llama.cpp rev-parse HEAD
    print("llama.cpp:", LLAMA_COMMIT[0], "| tam kimlik:", LLAMA_TAM[0])
    assert LLAMA_TAM[0].startswith(LLAMA_CPP_COMMIT), f"llama.cpp {LLAMA_CPP_ETIKET} değil: {LLAMA_TAM[0]}"
    QUANTIZE = '/content/llama.cpp/build/bin/llama-quantize'
    if not os.path.exists(QUANTIZE):
        print("llama-quantize derleniyor (~3-5 dk)...")
        !cmake -S /content/llama.cpp -B /content/llama.cpp/build -DGGML_CUDA=OFF -DLLAMA_CURL=OFF -DLLAMA_BUILD_TESTS=OFF -DLLAMA_BUILD_SERVER=OFF > /content/cmake.log 2>&1 || tail -20 /content/cmake.log
        !cmake --build /content/llama.cpp/build --target llama-quantize -j 8 > /content/build.log 2>&1 || tail -30 /content/build.log
    assert os.path.exists(QUANTIZE), "llama-quantize derlenemedi; /content/build.log dosyasına bak"

## 5) Taban model (~29,5 GB indirilir, ~5-10 dk; bu oturumda indirildiyse diskten okunur)

In [ ]:
if URET:
    from huggingface_hub import snapshot_download
    TABAN_HF = snapshot_download(TABAN_HF_ID)
    print("taban model:", TABAN_HF)

## 6) Dönüştürme adımı (iki model için aynı)

GGUF'a çevirme (f16) → Q4_K_M'e sıkıştırma → kimlik kaydı (`gguf_ortam_14b_<ad>.json`: llama.cpp sürümü, kaynak, boyut, sha256).
Yarım kalan dosya `.yarim` adıyla yazılır, bitince asıl adını alır; kopma sonrası yarım dosya "bitmiş" sanılmaz.
Model dosyaları (~9'ar GB) yalnız sunucuda kalır; Drive'a yedeklenmez (ikisi ~18 GB eder), kimlikleri (sha256) kayıtta durur.
Önceki bir oturumda aynı model için kayıt varsa ve yeni dosyanın parmak izi farklıysa eski kayıt `_onceki_` adıyla saklanır ve uyarı verilir.

In [ ]:
import datetime

def donustur(hf_dizin, f16, python):
    return subprocess.run([python, "/content/llama.cpp/convert_hf_to_gguf.py", hf_dizin, "--outtype", "f16", "--outfile", f16],
                          capture_output=True, text=True)

def ayri_ortam():
    # Donusturme betigi yeni transformers ile calismazsa: transformers<5 olan ayri bir sanal ortam (Colab ortami bozulmaz)
    py = "/content/donustur-venv/bin/python"
    if not os.path.exists(py):
        !python -m venv /content/donustur-venv || (pip install -q virtualenv && python -m virtualenv -q /content/donustur-venv)
        !/content/donustur-venv/bin/pip install -q torch --index-url https://download.pytorch.org/whl/cpu
        !/content/donustur-venv/bin/pip install -q "transformers>=4.45,<5" sentencepiece "protobuf>=4.21,<5" numpy safetensors pyyaml tqdm requests
    assert os.path.exists(py), "Ayrı dönüştürme ortamı kurulamadı (yukarıdaki çıktıyı Claude'a gönder)"
    return py

PYTHON = sys.executable

def gguf_uret(ad, hf_dizin, kaynak):
    global PYTHON
    f16 = f"/content/gguf/{ad}-f16.gguf"
    print(f"{ad}: GGUF'a çevriliyor (f16)...")
    r = donustur(hf_dizin, f16, PYTHON)
    if r.returncode != 0 and PYTHON == sys.executable:
        print("   dönüştürme bu ortamda başarısız, ayrı ortamda tekrar deneniyor:", r.stderr.strip().splitlines()[-1:])
        PYTHON = ayri_ortam()
        r = donustur(hf_dizin, f16, PYTHON)
    if r.returncode != 0:
        print(r.stdout[-1500:], r.stderr[-3000:])
        raise SystemExit(f"{ad}: GGUF dönüştürme başarısız (çıktıyı Claude'a gönder)")
    print(f"{ad}: Q4_K_M'e sıkıştırılıyor...")
    r = subprocess.run([QUANTIZE, f16, YEREL[ad] + ".yarim", "Q4_K_M"], capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stdout[-1500:], r.stderr[-1500:])
        raise SystemExit(f"{ad}: Q4_K_M sıkıştırma başarısız (çıktıyı Claude'a gönder)")
    os.replace(YEREL[ad] + ".yarim", YEREL[ad])
    os.remove(f16)
    bilgi = {"tarih": datetime.datetime.now().isoformat(timespec="minutes"), "model": ad, "llama_cpp": LLAMA_COMMIT[0],
             "llama_cpp_kimlik": LLAMA_TAM[0], "donusturme_python": PYTHON, "kart": KART, "kod": KOD[0], **kaynak,
             "dosya": os.path.basename(YEREL[ad]), "boyut": os.path.getsize(YEREL[ad]), "sha256": sha256(YEREL[ad])}
    if os.path.exists(BILGI[ad]):           # onceki oturumda ayni model icin uretilmis kayit (Drive'da)
        eski = json.load(open(BILGI[ad], encoding="utf-8"))
        if eski.get("sha256") == bilgi["sha256"]:
            print(f"{ad}: parmak izi önceki oturumda üretilen dosyayla aynı")
        else:
            sakla = BILGI[ad].replace(".json", f"_onceki_{eski.get('tarih', '')[:16].replace(':', '')}.json")
            os.replace(BILGI[ad], sakla)
            print(f"UYARI: {ad} önceki oturumda farklı bir dosya olarak üretilmiş; eski kayıt saklandı: {os.path.basename(sakla)}. "
                  "Yarım kalmış bir cevap dosyasına devam edilecekse Claude'a haber ver.")
    json.dump(bilgi, open(BILGI[ad], "w", encoding="utf-8"), ensure_ascii=False, indent=1)
    print(json.dumps(bilgi, ensure_ascii=False, indent=1))

## 7) `qwen14-taban`: birleştirme yok, aynı araçlarla GGUF + Q4_K_M (~20-25 dk)

In [ ]:
if TABAN in URET:
    gguf_uret(TABAN, TABAN_HF, {"taban_hf": TABAN_HF_ID, "taban_hf_surum": os.path.basename(TABAN_HF), "birlestirme": None})

## 8) `varchat-14b`: adaptörü tabana kalıcı ekle (işlemcide, bf16) + GGUF + Q4_K_M (~40-50 dk)

7B ile aynı birleştirme kodu (`PeftModel.from_pretrained(...).merge_and_unload()`). Birleştirmeden sonra indirilen taban model diskten
silinir: iki işi de (qwen14-taban dosyası ve birleştirme) bitmiştir; dönüştürme için ~30 GB yer açılır.

In [ ]:
if VARCHAT in URET:
    import gc, torch
    from transformers import AutoModelForCausalLM, AutoTokenizer
    from peft import PeftModel
    if not os.path.exists('/content/varchat-hf/config.json'):
        print("birleştiriliyor (CPU, ~20 dk)...")
        shutil.rmtree('/content/varchat-hf.yarim', ignore_errors=True)
        taban = AutoModelForCausalLM.from_pretrained(TABAN_HF, dtype=torch.bfloat16, device_map="cpu", low_cpu_mem_usage=True)
        birlesik = PeftModel.from_pretrained(taban, LORA_DIR).merge_and_unload()
        birlesik.save_pretrained('/content/varchat-hf.yarim', safe_serialization=True, max_shard_size="4GB")
        AutoTokenizer.from_pretrained(TABAN_HF).save_pretrained('/content/varchat-hf.yarim')
        if os.path.exists(f"{TABAN_HF}/generation_config.json"):
            shutil.copy(f"{TABAN_HF}/generation_config.json", '/content/varchat-hf.yarim/')
        shutil.rmtree('/content/varchat-hf', ignore_errors=True)
        os.replace('/content/varchat-hf.yarim', '/content/varchat-hf')
        del taban, birlesik
        gc.collect()
    print("birleştirildi -> /content/varchat-hf")
    if os.path.exists(YEREL[TABAN]):
        shutil.rmtree(os.path.dirname(os.path.dirname(TABAN_HF)), ignore_errors=True)
        print("indirilen taban model diskten silindi (qwen14-taban dosyası ve birleştirme bitti)")
    gguf_uret(VARCHAT, '/content/varchat-hf',
              {"taban_hf": TABAN_HF_ID, "taban_hf_surum": os.path.basename(TABAN_HF),
               "birlestirme": {"adaptor": LORA_DIR, "adaptor_sha256": ADAPTOR_SHA,
                               "adaptor_ayar": {k: LORA_AYAR.get(k) for k in ("r", "lora_alpha", "lora_dropout", "peft_version")}}})
    shutil.rmtree('/content/varchat-hf', ignore_errors=True)

## 9) Ollama 0.35.1'i kur, başlat; hazır `qwen2.5:14b`'nin sohbet şablonunu al

Şablon (TEMPLATE + SYSTEM satırı) hazır `qwen2.5:7b`'ninkiyle karşılaştırılır ve sonuç `ortam_14b_<tarih>.json`'a yazılır.
Farklı çıksa da 14B hücreleri hazır 14B'nin şablonunu kullanır (Ekim'de ölçülen hazır 14B de onunla cevap yazdı).

In [ ]:
!apt-get -qq update > /dev/null && apt-get -qq install -y zstd pciutils > /dev/null
!curl -fsSL https://ollama.com/install.sh | OLLAMA_VERSION={OLLAMA_SURUM} sh > /dev/null

import time, requests

def ollama_hazir():
    try:
        return requests.get('http://127.0.0.1:11434/api/version', timeout=2).json()
    except Exception:
        return None

if not ollama_hazir():
    subprocess.Popen(['ollama', 'serve'], stdout=open('/content/ollama.log', 'w'), stderr=subprocess.STDOUT)
    for _ in range(60):
        time.sleep(1)
        if ollama_hazir():
            break
SURUM = (ollama_hazir() or {}).get("version")
print("Ollama:", SURUM)
assert SURUM == OLLAMA_SURUM, f"Ollama {OLLAMA_SURUM} değil ({SURUM}); /content/ollama.log ve bu çıktıyı Claude'a gönder"

!ollama pull {HAZIR}
ms = !ollama show {HAZIR} --modelfile
HAZIR_MF = "\n".join(ms)
satir = HAZIR_MF.split('\nLICENSE """')[0].split("\n")
govde = "\n".join(satir[next(k for k, s in enumerate(satir) if s.startswith("FROM ")) + 1:]).strip()
SABLON_SHA = hashlib.sha256(govde.encode()).hexdigest()
print("sohbet şablonu, hazır qwen2.5:7b'ninkiyle:", "AYNI" if SABLON_SHA == SABLON_7B_SHA256 else "FARKLI (not edilir)")
with open(f"{SONUC}/modelfile_{HAZIR.replace(':', '-')}.txt", "w", encoding="utf-8", newline="\n") as f:
    f.write(HAZIR_MF)

## 10) İki model dosyasını Ollama'ya yükle (hazır 14B'nin şablonu; yalnız `FROM` satırı değişir)

In [ ]:
import re, ollama
for ad in (TABAN, VARCHAT):
    assert os.path.exists(YEREL[ad]), f"{YEREL[ad]} yok: 7. ya da 8. bölüm tamamlanmamış"
    with open(f"/content/Modelfile.{ad}", "w", newline="\n") as f:
        f.write(re.sub(r"(?m)^FROM .*$", f"FROM {YEREL[ad]}", HAZIR_MF))
    !ollama create {ad} -f /content/Modelfile.{ad}
    try:
        ollama.show(ad)
    except ollama.ResponseError as e:
        raise SystemExit(f"{ad} Ollama'ya yüklenemedi: {e}. Yukarıdaki 'ollama create' çıktısını Claude'a gönder.")
    print(f"{ad}: Ollama'da hazır")

# Kisa kontrol: ayni soru, uc model (yalnizca goz kontrolu; sayisal olcum degil)
soru = [{"role": "system", "content": "Yanıtın DAİMA Türkçe olmalı."}, {"role": "user", "content": "BRAF V600E nedir? Bir cümleyle anlat."}]
for ad in (HAZIR, TABAN, VARCHAT):
    r = ollama.chat(model=ad, messages=soru, options={"temperature": 0, "seed": 42, "num_predict": 60, "num_ctx": 8192})
    print(f"{ad:13} {r['message']['content'][:160]!r}")
ps_cikti = !ollama ps
print("\n".join(ps_cikti))
if not any("100% GPU" in s for s in ps_cikti[1:]):
    print("\nUYARI: model tamamen GPU'da değil (kısmen işlemcide); cevaplar çok yavaş üretilir. Bu çıktıyı Claude'a gönder.")

liste = !ollama list
ORTAM = {"tarih": datetime.datetime.now().isoformat(timespec="minutes"), "gpu": KART, "ram_gb": round(RAM_GB),
         "gpu_bos_gb_baslangic": round(BOS_GPU_GB, 1), "ollama": ollama_hazir(), "modeller": list(liste), "ollama_ps": list(ps_cikti),
         "kod": KOD[0], "sablon_sha256": SABLON_SHA, "sablon_qwen2.5-7b_ile_ayni": SABLON_SHA == SABLON_7B_SHA256}
json.dump(ORTAM, open(f"{SONUC}/ortam_14b_{datetime.date.today().isoformat()}.json", "w", encoding="utf-8"), ensure_ascii=False, indent=1)
print(json.dumps(ORTAM, ensure_ascii=False, indent=1))

## 11) `qwen14-taban`: 56 cevap (~15-25 dk)

In [ ]:
!python -u evaluation/e04_generate.py {TABAN}

## 12) `varchat-14b`: 56 cevap (~15-25 dk)

In [ ]:
!python -u evaluation/e04_generate.py {VARCHAT}

## 13) Cevap dosyalarının kontrolü + yargıçsız ön bakış

Her yeni dosyada 56 girdi, aynı donmuş kaynak, aynı ayarlar ve hazır 14B ile aynı istemler olmalı; değilse hata verir.
Tablodaki cümle ve atıfsız cümle sayıları m01'in kendi denetiminden gelir (yargıç yok; e05/e13'ün iddia sayımından biraz farklı olabilir).

In [ ]:
import pandas as pd

def uretim_dosyasi(model):
    d = glob.glob(f"{SONUC}/uretim_{model.replace(':', '-')}_*.jsonl")
    return max(d, key=os.path.getmtime) if d else None

tablo = []
for ad in (HAZIR, TABAN, VARCHAT):
    yol = f"{SONUC}/{KARSI}" if ad == HAZIR else uretim_dosyasi(ad)
    assert yol, f"{ad}: cevap dosyası yok"
    rows = [json.loads(l) for l in open(yol, encoding="utf-8") if l.strip()]
    if ad != HAZIR:
        assert len(rows) == 56 and {r["girdi"] for r in rows} == set(karsi), f"{ad}: 56 girdi eksiksiz değil ({len(rows)} satır)"
        assert all(r["snapshot"] == SNAPSHOT and r["ayarlar"] == AYARLAR_BEKLENEN for r in rows), f"{ad}: kaynak ya da ayar farklı"
        assert all((r.get("sistem"), r.get("soru")) == (karsi[r["girdi"]]["sistem"], karsi[r["girdi"]]["soru"]) for r in rows), \
            f"{ad}: istem hazır 14B'ninkinden farklı"
    d = [r.get("denetim") or {} for r in rows]
    cumle = sum(x.get("cumle") or 0 for x in d)
    atifsiz = sum(x.get("atifsiz_cumle") or 0 for x in d)
    tablo.append({"model": ad, "dosya": os.path.basename(yol), "cevap": sum(1 for r in rows if r.get("cevap")),
                  "yalnız ret": sum(1 for r in rows if (r.get("cevap") or "").strip() == e04.app.RET_CEVAP),
                  "cümle": cumle, "atıfsız cümle oranı": round(atifsiz / cumle, 3) if cumle else None,
                  "dil bozuk": sum(1 for x in d if x.get("bozuk")), "kesilen": sum(1 for x in d if x.get("bitis") == "length"),
                  "üretim hatası": sum(1 for r in rows if r.get("hata")),
                  "ort. süre (sn)": round(sum(r.get("sure_s") or 0 for r in rows) / len(rows), 1)})
display(pd.DataFrame(tablo))

## 14) Sonuçları bilgisayarına al

Drive'daki sonuç klasöründe repoda olmayan dosyalar (yeni dosyalar) bulunur ve tek zip'e konur: `varchat-14b/olcum_14b_sonuclari.zip`.
Repodaki sonuç dosyalarının Drive kopyaları git'in kayıtlarıyla bayt bayt karşılaştırılır; biri eksik ya da farklıysa defter durur.
Sonra Drive bağlantısı kapatılır (bekleyen yazmalar bu sırada Drive'a gönderilir).

**Nasıl indirilir:** drive.google.com → `varchat-14b` → **`olcum_14b_sonuclari.zip`** → sağ tık → İndir → projede `evaluation/results/`
klasörüne aç. Zip'te yalnız yeni dosyalar var; yereldeki hiçbir dosyanın üzerine yazılmaz.
Sonra **Çalışma zamanı → Bağlantıyı kes ve çalışma zamanını sil** ve Claude'a haber ver: puanlama yerelde `evaluation/e13_judge_14b.py` ile yapılır.

In [ ]:
import zipfile

def git_izi(yol):
    # git'in dosya kimligi (blob sha1); repodaki surumle bayt bayt ayni dosya ayni izi verir
    veri = open(yol, "rb").read()
    return hashlib.sha1(b"blob %d\0" % len(veri) + veri).hexdigest()

agac = !git -C /content/varchat ls-tree -r origin/main -- evaluation/results
repo = {s.split("\t", 1)[1].split("/")[-1]: s.split()[2] for s in agac if "\t" in s}
assert repo, "git kaydı okunamadı (ls-tree boş): çıktıyı Claude'a gönder"
degisen = [ad for ad, iz in repo.items() if not os.path.exists(f"{SONUC}/{ad}") or git_izi(f"{SONUC}/{ad}") != iz]
assert not degisen, f"Repodaki sonuç dosyalarından biri Drive kopyasında eksik ya da farklı (olmamalı): {degisen}"
yeni = sorted(a for a in os.listdir(SONUC) if os.path.isfile(f"{SONUC}/{a}") and a not in repo)
for onek in (f"uretim_{TABAN}_", f"uretim_{VARCHAT}_", f"gguf_ortam_14b_{TABAN}", f"gguf_ortam_14b_{VARCHAT}", "ortam_14b_", "modelfile_"):
    assert any(a.startswith(onek) for a in yeni), f"{onek}* dosyası yok: önceki bölümlerden biri bitmemiş"

yerel_zip = "/content/olcum_14b_sonuclari.zip"       # once sunucunun diskinde, sonra tek seferde Drive'a
with zipfile.ZipFile(yerel_zip, "w", zipfile.ZIP_DEFLATED) as z:
    for a in yeni:
        z.write(f"{SONUC}/{a}", a)
shutil.copy(yerel_zip, ZIP)
print(f"{ZIP}: {os.path.getsize(ZIP) / 1e6:.1f} MB | repodaki {len(repo)} dosya değişmemiş")
for a in yeni:
    print("  ", a)
try:
    drive.flush_and_unmount()                # bekleyen yazmalari Drive'a gonderir, sonra baglantiyi kapatir
    print("\nDrive'a gönderildi; Drive bağlantısı kapatıldı.")
except Exception as e:
    print("\nUYARI: Drive bağlantısı düzgün kapatılamadı:", e, "| Çalışma zamanını silmeden önce zip'i drive.google.com'da gör.")
print("İndir: drive.google.com → varchat-14b → olcum_14b_sonuclari.zip → sağ tık → İndir → projede evaluation/results/ içine aç.")